# 预处理：为三条模型路线准备共同数据

**问题：** 三条路线需要比较同一批比赛。如果各自清洗原始数据，容易出现样本和历史记录不一致。

**做法：** 在这里完成清洗、Elo 补全、近期与交锋特征、时间划分以及历史序列生成。模型 notebook 直接读取输出，不互相执行。

| 输入 | 输出 | 用途 |
|---|---|---|
| `data/raw/Matches.csv`、`EloRatings.csv` | `data/processed/cleaned.csv` | 清洗后的比赛表 |
| 全部有效赛果与 cleaned 样本 | `baseline_features.csv` | B/F/H/Q、标签、赔率概率和时间划分 |
| 全部有效赛果与同一批样本 | `history_sequences.npz` | 时间衰减与 LSTM 共用的逐场历史 |

从上到下运行。本 notebook 不训练预测模型。代码和输出用英文，解释用中文。


## 1. 读取原始数据

先显示缺失比例，了解哪些列可用。后面按用途处理缺失，不对所有列统一填 0。


In [1]:
from pathlib import Path
from collections import defaultdict, deque
import unicodedata
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "data/raw/Matches.csv").exists():
    ROOT = ROOT.parent
PROCESSED = ROOT / "data/processed"
PROCESSED.mkdir(parents=True, exist_ok=True)
COUNTRY = {d: d for d in "ARG AUT BRA CHN DEN FIN IRL JAP MEX NOR POL ROM RUS SUI SWE USA".split()}
COUNTRY.update(
    {
        d: c
        for c, divisions in {
            "ENG": "E0 E1 E2 E3 EC",
            "SCO": "SC0 SC1 SC2 SC3",
            "GER": "D1 D2",
            "FRA": "F1 F2",
            "ITA": "I1 I2",
            "ESP": "SP1 SP2",
            "BEL": "B1",
            "GRE": "G1",
            "NED": "N1",
            "POR": "P1",
            "TUR": "T1",
        }.items()
        for d in divisions.split()
    }
)
CORE = ["Division", "MatchDate", "HomeTeam", "AwayTeam", "FTHome", "FTAway", "FTResult"]
ODDS = ["OddHome", "OddDraw", "OddAway"]
TRAIN_END, VALID_END, TEST_END = ("2020-12-31", "2022-12-31", "2024-12-31")


def norm(value):
    return " ".join(unicodedata.normalize("NFKC", str(value)).split()) if pd.notna(value) else value


raw = pd.read_csv(
    ROOT / "data/raw/Matches.csv", na_values=[" ", "na", "n/a", "null", "none"], low_memory=False
)
elo = pd.read_csv(ROOT / "data/raw/EloRatings.csv")
print("Raw matches:", len(raw), "| Elo records:", len(elo))
missing = raw.isna().mean().mul(100).rename("missing_percent").rename_axis("column").reset_index()
display(missing.loc[missing.missing_percent.gt(0)].round(2))


Raw matches: 238858 | Elo records: 273972


,column,missing_percent
2,MatchTime,55.05
5,HomeElo,37.24
6,AwayElo,37.27
7,Form3Home,0.64
8,Form5Home,0.64
9,Form3Away,0.64
10,Form5Away,0.64
11,FTHome,0.00
12,FTAway,0.00
13,FTResult,0.00


## 2. 保留有效赛果历史

身份、日期或全场赛果缺失的记录删除；比分必须为非负整数、与胜平负一致，双方不能相同。有效历史暂不要求 Elo 或赔率完整，因为这些比赛的赛果仍然已知。球队按国家和规范化队名识别，升降级不截断历史。


In [2]:
matches = raw.copy()
matches.insert(0, "SourceRow", np.arange(2, len(matches) + 2))
for col in ["Division", "HomeTeam", "AwayTeam"]:
    matches[col] = matches[col].map(norm)
matches["MatchDate"] = pd.to_datetime(matches.MatchDate, errors="coerce")
for col in ["FTHome", "FTAway"]:
    matches[col] = pd.to_numeric(matches[col], errors="coerce")
scores = matches[["FTHome", "FTAway"]]
expected = np.where(
    matches.FTHome > matches.FTAway, "H", np.where(matches.FTHome < matches.FTAway, "A", "D")
)
valid = (
    matches[CORE].notna().all(axis=1)
    & np.isfinite(scores).all(axis=1)
    & scores.ge(0).all(axis=1)
    & scores.mod(1).eq(0).all(axis=1)
    & matches.FTResult.eq(expected)
    & matches.HomeTeam.ne(matches.AwayTeam)
)
valid_history = matches.loc[valid].sort_values(["MatchDate", "SourceRow"]).reset_index(drop=True)
valid_history["Country"] = valid_history.Division.map(COUNTRY)
assert valid_history.Country.notna().all(), "Unknown division: update COUNTRY."
assert not valid_history.duplicated(
    ["Division", "MatchDate", "HomeTeam", "AwayTeam"]
).any(), "Duplicate matches."
print(
    "Valid historical matches:", len(valid_history), "| Removed invalid rows:", int((~valid).sum())
)


Valid historical matches: 238854 | Removed invalid rows: 4


## 3. 只补全缺失的 Elo

保留已有值。缺失侧使用同国家、同球队、**严格早于比赛日**的最近评分，不设三个月上限。当天和未来评分不参与；仍缺 Elo 的比赛不能作为预测样本，但继续进入历史池。


In [3]:
elo["date"] = pd.to_datetime(elo.date, errors="coerce")
elo["elo"] = pd.to_numeric(elo.elo, errors="coerce")
elo["club"] = elo.club.map(norm)
elo["country"] = elo.country.map(norm)
elo = (
    elo.dropna(subset=["date", "club", "country", "elo"])
    .drop_duplicates(["country", "club", "date"])
    .sort_values("date")
)
history = valid_history.copy()
elo_summary = []
for side in ["Home", "Away"]:
    col = side + "Elo"
    history[col] = pd.to_numeric(history[col], errors="coerce")
    missing_mask = history[col].isna()
    left = (
        history.loc[missing_mask, ["MatchDate", "Country", side + "Team"]]
        .rename(columns={side + "Team": "club", "Country": "country"})
        .assign(row=lambda d: d.index)
        .sort_values("MatchDate")
    )
    found = pd.merge_asof(
        left,
        elo,
        left_on="MatchDate",
        right_on="date",
        by=["country", "club"],
        direction="backward",
        allow_exact_matches=False,
    )
    history.loc[found.row, col] = found.elo.to_numpy()
    elo_summary.append(
        {
            "column": col,
            "missing_before": int(missing_mask.sum()),
            "missing_after": int(history[col].isna().sum()),
        }
    )
have_elo = history[["HomeElo", "AwayElo"]].notna().all(axis=1)
display(pd.DataFrame(elo_summary))


,column,missing_before,missing_after
0,HomeElo,88959,78336
1,AwayElo,89028,78369


## 4. 重算 Form，筛选共同样本

球队每场积分为胜 3、平 1、负 0。赛前 Form 定义为

$$\mathrm{Form}k(t)=\sum_{i=1}^{k}\mathrm{points}_{t-i},\quad k\in\{3,5\}.$$

只使用严格更早日期的比赛，双方最近五场必须齐全。删除缺失、无穷或不大于 1 的普通胜平负赔率。

使用明确的列清单保存 21 列，不依赖原表列的位置。技术统计、额外盘口和 C_* 列不进入本版。半场结果可缺失，因为模型不使用。


In [4]:
past = defaultdict(lambda: deque(maxlen=5))
form = {
    c: np.full(len(history), np.nan) for c in ["Form3Home", "Form5Home", "Form3Away", "Form5Away"]
}
for day, batch in history.groupby("MatchDate", sort=True):
    for r in batch.itertuples():
        for side in ["Home", "Away"]:
            prior = list(past[r.Country, getattr(r, side + "Team")])
            for k in (3, 5):
                if len(prior) >= k:
                    form[f"Form{k}{side}"][r.Index] = sum(prior[-k:])
    for r in batch.itertuples():
        for side, win in [("Home", "H"), ("Away", "A")]:
            pts = 1 if r.FTResult == "D" else 3 if r.FTResult == win else 0
            past[r.Country, getattr(r, side + "Team")].append(pts)
for c, v in form.items():
    history[c] = pd.array(v, dtype="Int64")
completed = history.loc[have_elo].copy()
form_ok = completed[["Form5Home", "Form5Away"]].notna().all(axis=1)
removed_form = int((~form_ok).sum())
completed = completed.loc[form_ok].copy()
odds = completed[ODDS].apply(pd.to_numeric, errors="coerce")
odds_ok = np.isfinite(odds).all(axis=1) & odds.gt(1).all(axis=1)
removed_odds = int((~odds_ok).sum())
columns = [
    "SourceRow",
    "Division",
    "MatchDate",
    "HomeTeam",
    "AwayTeam",
    "HomeElo",
    "AwayElo",
    "Form3Home",
    "Form5Home",
    "Form3Away",
    "Form5Away",
    "FTHome",
    "FTAway",
    "FTResult",
    "HTHome",
    "HTAway",
    "HTResult",
    "OddHome",
    "OddDraw",
    "OddAway",
    "Country",
]
cleaned = completed.loc[odds_ok, columns].copy().reset_index(drop=True)
cleaned[ODDS] = odds.loc[odds_ok].to_numpy()
cleaned.to_csv(PROCESSED / "cleaned.csv", index=False, date_format="%Y-%m-%d")
summary = pd.DataFrame(
    [
        {"stage": "raw", "matches": len(raw)},
        {"stage": "valid_history", "matches": len(history)},
        {"stage": "both_elo", "matches": int(have_elo.sum())},
        {"stage": "complete_last_5", "matches": len(completed)},
        {"stage": "valid_odds_cleaned", "matches": len(cleaned)},
    ]
)
print(
    "Cleaned:",
    cleaned.shape,
    "| Removed incomplete Form5:",
    removed_form,
    "| Removed invalid odds:",
    removed_odds,
)
display(summary)


Cleaned: (141733, 21) | Removed incomplete Form5: 2170 | Removed invalid odds: 2000


,stage,matches
0,raw,238858
1,valid_history,238854
2,both_elo,145903
3,complete_last_5,143733
4,valid_odds_cleaned,141733


## 5. 明确每个输入特征

**问题:** 近 5 场、交锋和赔率的原始尺度不同，主客方向也不一致。直接把比分或赔率原值送进模型，无法比较近期和交锋哪一块有用。

**解决办法:** 全部改成赛前的差或比例。近 5 场用各队自己的视角，交锋用本场主队的视角。没有交锋时，H 三项记 0，并用 `h2h_missing` 标出。赔率先变成概率，再取相对平局的对数比。

| 组 | 列名 | 含义 |
|---|---|---|
| B | Division | 联赛，训练期独热编码 |
| B | elo_diff | 当前主队Elo减客队Elo |
| B | h2h_count | 此前全部直接交锋次数 |
| B | h2h_missing | 没有交锋为1，否则0 |
| F | recent_ppg_diff | 双方各自最近5场场均积分之差 |
| F | recent_gd_diff | 双方各自最近5场场均净胜球之差 |
| F | recent_draw_rate_mean | 双方各自最近5场平局比例的平均值 |
| H | h2h_ppg_diff | 直接交锋中，本场主队积分减本场客队积分的平均值 |
| H | h2h_gd | 直接交锋中，本场主队进球减失球的平均值；不重复做一次差 |
| H | h2h_draw_rate | 直接交锋中平局的比例 |
| Q | q_log_H_D、q_log_A_D | 归一化赔率概率的两个对数比，仅用于包含赔率输入的模型 |

无交锋时，H三项为0，B里的无交锋标记为1。当前比赛的比分、赛果、射门和半场等不进入X。

近期特征的主客队是当前比赛双方；每场历史成绩都转换到该球队视角。交锋特征始终转换到本场主队视角。同一比赛日先用已有历史生成全部特征，再把当日赛果写入历史。

近期窗口为双方各自严格早于比赛日的最近 5 场，对手不限。积分按该队赛果计，胜 3、平 1、负 0。$d_i$ 表示该场是否平局。

$$
\mathrm{recent\_ppg\_diff}=\bar p_H-\bar p_A,\qquad
\bar p=\frac{1}{5}\sum_{i=1}^{5}\mathrm{pts}_i
$$

$$
\mathrm{recent\_gd\_diff}=\bar g_H-\bar g_A,\qquad
\mathrm{recent\_draw\_rate\_mean}=\frac{\bar d_H+\bar d_A}{2}
$$

交锋为两队严格早于比赛日的全部直接比赛，场数 $n=\mathrm{h2h\_count}$。积分差与净胜球取本场主队视角。$n=0$ 时三项为 0，且 $\mathrm{h2h\_missing}=1$。

$$
\mathrm{h2h\_ppg\_diff}=\frac{1}{n}\sum_{t=1}^{n}(\mathrm{pts}_{\mathrm{Home},t}-\mathrm{pts}_{\mathrm{Away},t})
$$

$$
\mathrm{h2h\_gd}=\frac{1}{n}\sum_{t=1}^{n}(\mathrm{GF}_{\mathrm{Home},t}-\mathrm{GA}_{\mathrm{Home},t}),\qquad
\mathrm{h2h\_draw\_rate}=\frac{1}{n}\sum_{t=1}^{n}d_t
$$

$$
\mathrm{elo\_diff}=\mathrm{HomeElo}-\mathrm{AwayElo}
$$

$$
q_k=\frac{1/\mathrm{Odd}_k}{\sum_{j\in\{H,D,A\}}1/\mathrm{Odd}_j},\qquad
\mathrm{q\_log\_H\_D}=\log\frac{q_H}{q_D},\qquad
\mathrm{q\_log\_A\_D}=\log\frac{q_A}{q_D}
$$

预处理写出 `data/processed/baseline_features.csv`。


In [5]:
def make_history_features(history_frame, wanted_ids):
    recent = defaultdict(lambda: deque(maxlen=5))
    pairs = {}
    rows = []
    for day, batch in history_frame.groupby("MatchDate", sort=True):
        for r in batch.itertuples():
            if r.SourceRow not in wanted_ids:
                continue
            home_recent = list(recent[r.Country, r.HomeTeam])
            away_recent = list(recent[r.Country, r.AwayTeam])
            means = []
            for records in (home_recent, away_recent):
                if len(records) == 5:
                    means.append(
                        (
                            sum((x[1] for x in records)) / 5,
                            sum((x[2] for x in records)) / 5,
                            sum((x[3] for x in records)) / 5,
                        )
                    )
                else:
                    means.append((np.nan, np.nan, np.nan))
            first, second = sorted([r.HomeTeam, r.AwayTeam])
            n, points_sum, gd_sum, draws, _ = pairs.get(
                (r.Country, first, second), (0, 0, 0, 0, None)
            )
            sign = 1 if r.HomeTeam == first else -1
            rows.append(
                {
                    "SourceRow": r.SourceRow,
                    "recent_ppg_diff": means[0][0] - means[1][0],
                    "recent_gd_diff": means[0][1] - means[1][1],
                    "recent_draw_rate_mean": (means[0][2] + means[1][2]) / 2,
                    "home_recent_points_sum": (
                        None if len(home_recent) < 5 else sum((x[1] for x in home_recent))
                    ),
                    "away_recent_points_sum": (
                        None if len(away_recent) < 5 else sum((x[1] for x in away_recent))
                    ),
                    "h2h_count": n,
                    "h2h_missing": int(n == 0),
                    "h2h_ppg_diff": sign * points_sum / n if n else 0.0,
                    "h2h_gd": sign * gd_sum / n if n else 0.0,
                    "h2h_draw_rate": draws / n if n else 0.0,
                }
            )
        for r in batch.itertuples():
            home_points = 3 if r.FTResult == "H" else 1 if r.FTResult == "D" else 0
            away_points = 3 if r.FTResult == "A" else 1 if r.FTResult == "D" else 0
            gd, draw = (r.FTHome - r.FTAway, int(r.FTResult == "D"))
            recent[r.Country, r.HomeTeam].append((day, home_points, gd, draw))
            recent[r.Country, r.AwayTeam].append((day, away_points, -gd, draw))
            first, second = sorted([r.HomeTeam, r.AwayTeam])
            key, sign = ((r.Country, first, second), 1 if r.HomeTeam == first else -1)
            n, ps, gs, ds, _ = pairs.get(key, (0, 0, 0, 0, None))
            pairs[key] = (
                n + 1,
                ps + sign * (home_points - away_points),
                gs + sign * gd,
                ds + draw,
                day,
            )
    return pd.DataFrame(rows)


history = make_history_features(valid_history, set(cleaned.SourceRow))
joined = (
    cleaned.merge(history, on="SourceRow", validate="one_to_one")
    .sort_values(["MatchDate", "SourceRow"])
    .reset_index(drop=True)
)
joined["elo_diff"] = joined.HomeElo - joined.AwayElo
joined["year"] = joined.MatchDate.dt.year
joined["y"] = joined.FTResult.map({"H": 0, "D": 1, "A": 2}).astype(int)
q = 1 / joined[["OddHome", "OddDraw", "OddAway"]].to_numpy(float)
q /= q.sum(axis=1, keepdims=True)
joined[["q_H", "q_D", "q_A"]] = q
joined["q_log_H_D"] = np.log(q[:, 0] / q[:, 1])
joined["q_log_A_D"] = np.log(q[:, 2] / q[:, 1])
B = ["elo_diff", "h2h_count", "h2h_missing"]
F = ["recent_ppg_diff", "recent_gd_diff", "recent_draw_rate_mean"]
H = ["h2h_ppg_diff", "h2h_gd", "h2h_draw_rate"]
Q = ["q_log_H_D", "q_log_A_D"]
keep = [
    "SourceRow",
    "MatchDate",
    "year",
    "Country",
    "Division",
    "HomeTeam",
    "AwayTeam",
    "FTResult",
    "y",
]
features = joined[
    keep + B + F + H + Q + ["q_H", "q_D", "q_A", "home_recent_points_sum", "away_recent_points_sum"]
].copy()
features["split"] = np.select(
    [
        features.MatchDate.le(TRAIN_END),
        features.MatchDate.le(VALID_END),
        features.MatchDate.le(TEST_END),
    ],
    ["train", "validation", "test"],
    default="excluded_after_2024",
)
features.to_csv(
    PROCESSED / "baseline_features.csv", index=False, encoding="utf-8-sig", date_format="%Y-%m-%d"
)
print("features:", features.shape)
display(features[["MatchDate", "HomeTeam", "AwayTeam"] + B + F + H].head())


features: (141733, 26)


,MatchDate,HomeTeam,AwayTeam,elo_diff,h2h_count,h2h_missing,recent_ppg_diff,recent_gd_diff,recent_draw_rate_mean,h2h_ppg_diff,h2h_gd,h2h_draw_rate
0,2000-09-05,Monaco,Marseille,87.64,0,1,0.2,0.0,0.3,0.0,0.0,0.0
1,2000-09-06,Bastia,Lille,11.73,0,1,-0.2,-0.6,0.1,0.0,0.0,0.0
2,2000-09-06,Guingamp,Toulouse,13.41,0,1,-0.4,-0.6,0.4,0.0,0.0,0.0
3,2000-09-06,Lens,Auxerre,108.50,0,1,0.4,0.6,0.2,0.0,0.0,0.0
4,2000-09-06,Metz,Strasbourg,65.38,0,1,0.4,1.8,0.2,0.0,0.0,0.0


## 6. 生成逐场历史，统一时间划分

| 序列 | 形状 | 每个时间步 |
|---|---|---|
| 近期 | `[比赛, 2, 5, 4]` | 该队积分、净胜球、平局标记、距本场天数 |
| 交锋 | `[比赛, 最长交锋数, 4]` | 本场主队减客队的积分差、主队净胜球、平局标记、距本场天数 |

历史按旧→新排列。交锋保留全部记录，短序列末尾补 0，另存实际长度；补齐不算真实比赛。没有交锋的统计和表示为 0，B 中有 `h2h_missing`。

历史生成不拟合参数，允许保存测试期赛前已知记录。模型只在训练期学习标准化和参数；同一测试年中，较早已结束的比赛可以进入较晚比赛的历史。


In [6]:
def build_sequences(history, targets):
    lookup = {int(r): i for i, r in enumerate(targets.SourceRow)}
    recent = defaultdict(lambda: deque(maxlen=5))
    pairs = defaultdict(list)
    recent_out = np.zeros((len(targets), 2, 5, 4), dtype=np.float32)
    pair_out = [None] * len(targets)
    for day, batch in history.groupby("MatchDate", sort=True):
        ordinal = day.toordinal()
        for r in batch.itertuples():
            if r.SourceRow not in lookup:
                continue
            i = lookup[r.SourceRow]
            for side, team in enumerate([r.HomeTeam, r.AwayTeam]):
                records = np.array(recent[r.Country, team], dtype=np.float32)
                assert records.shape == (5, 4), "Each sample needs exactly five prior matches."
                records[:, 3] = ordinal - records[:, 3]
                assert (records[:, 3] > 0).all()
                recent_out[i, side] = records
            first, second = sorted([r.HomeTeam, r.AwayTeam])
            records = np.array(pairs[r.Country, first, second], dtype=np.float32).reshape(-1, 4)
            sign = 1 if r.HomeTeam == first else -1
            records[:, :2] *= sign
            records[:, 3] = ordinal - records[:, 3]
            assert (records[:, 3] > 0).all()
            pair_out[i] = records
        for r in batch.itertuples():
            hp = 3 if r.FTResult == "H" else 1 if r.FTResult == "D" else 0
            ap = 3 if r.FTResult == "A" else 1 if r.FTResult == "D" else 0
            gd, draw = (r.FTHome - r.FTAway, int(r.FTResult == "D"))
            recent[r.Country, r.HomeTeam].append((hp, gd, draw, ordinal))
            recent[r.Country, r.AwayTeam].append((ap, -gd, draw, ordinal))
            first, second = sorted([r.HomeTeam, r.AwayTeam])
            sign = 1 if r.HomeTeam == first else -1
            pairs[r.Country, first, second].append((sign * (hp - ap), sign * gd, draw, ordinal))
    lengths = np.array([len(x) for x in pair_out], dtype=np.int64)
    padded = np.zeros((len(targets), max(1, int(lengths.max())), 4), dtype=np.float32)
    for i, records in enumerate(pair_out):
        padded[i, : len(records)] = records
    return (recent_out, padded, lengths)


recent_seq, h2h_seq, h2h_lengths = build_sequences(valid_history, features)
np.savez_compressed(
    PROCESSED / "history_sequences.npz",
    SourceRow=features.SourceRow.to_numpy(),
    recent=recent_seq,
    h2h=h2h_seq,
    h2h_lengths=h2h_lengths,
)
split_table = (
    features.groupby("split", sort=False)
    .agg(
        matches=("SourceRow", "size"),
        first_date=("MatchDate", "min"),
        last_date=("MatchDate", "max"),
        divisions=("Division", "nunique"),
    )
    .reset_index()
)
print("Recent tensor:", recent_seq.shape, "| Head-to-head tensor:", h2h_seq.shape)
display(split_table)


Recent tensor: (141733, 2, 5, 4) | Head-to-head tensor: (141733, 85, 4)


,split,matches,first_date,last_date,divisions
0,train,106288,2000-09-05,2020-12-31,29
1,validation,12598,2021-01-01,2022-12-31,29
2,test,11772,2023-01-01,2024-12-30,29
3,excluded_after_2024,11075,2025-01-01,2026-09-03,22


## 7. 更新总报告的数据部分

只更新对应章节，保留三条模型路线已有的结果。


In [7]:
def md_table(frame):
    rows = [
        "| " + " | ".join(map(str, frame.columns)) + " |",
        "| " + " | ".join(["---"] * len(frame.columns)) + " |",
    ]
    for values in frame.itertuples(index=False, name=None):
        rows.append(
            "| "
            + " | ".join(
                (f"{v:.6f}" if isinstance(v, (float, np.floating)) else str(v) for v in values)
            )
            + " |"
        )
    return "\n".join(rows)


def update_report(section, text):
    path = ROOT / "reports/reports.md"
    path.parent.mkdir(exist_ok=True)
    body = path.read_text(encoding="utf-8") if path.exists() else "# 足球比赛分析：运行结果\n\n"
    start, end = (f"<!-- {section}:start -->", f"<!-- {section}:end -->")
    block = start + "\n" + text.strip() + "\n" + end
    if start in body:
        before, remainder = body.split(start, 1)
        _, after = remainder.split(end, 1)
        body = before + block + after
    else:
        body = body.rstrip() + "\n\n" + block + "\n"
    path.write_text(body, encoding="utf-8")
    print("Updated reports/reports.md:", section)


text = "## 共同数据与处理规则\n\n"
text += "只使用项目中的 Matches.csv 和 EloRatings.csv，日期按文件记录使用。已有 Elo 保留；缺失时按国家和球队查找严格早于比赛日的最近评分，不限制间隔，仍缺失则删除。\n\n"
text += md_table(summary) + "\n\n" + md_table(split_table) + "\n\n"
text += "近期固定最近五场，不区分主客场；交锋包含此前全部直接交锋。计算历史时使用全部有效赛果，包括没有 Elo 或赔率、未进入 cleaned 的比赛。同一天先生成特征，再更新赛果。三项普通赔率均须有限且大于 1。\n\n"
text += "原始 Form 列重算；半场列只保存在 cleaned 中，不进入模型。cleaned 为比赛表，baseline_features 为模型公共特征表；history_sequences 为未拟合的逐场历史缓存。\n\n"
text += "模型先在截至 2020 年的数据拟合，2021—2022 年选择参数，再在截至 2022 年的数据最终拟合，评价 2023—2024 年。2025 年及以后保留但不参与当前拟合和评分。数值标准化仅在当前拟合集学习。\n\n"
text += "这些测试年已在此前分析中查看；以下为后续比较。样本要求 Elo、完整近五场和有效赔率，结论不能推广到全部原始比赛。原有 Elo 和赔率的独立记录时间没有另行验证。"
update_report("data", text)


Updated reports/reports.md: data
